In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from skimage.feature import hog
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

warnings.filterwarnings('ignore')

In [ ]:
# LOAD & PREPROCESS DATA

print("Loading dataset...")
train = pd.read_csv('emnist-letters-train.csv', header=None)

# Ambil 2600 sampel balanced (100 per kelas)
samples_per_class = 100
balanced_data = []
for label in range(1, 27):
    class_data = train[train[0] == label].sample(n=samples_per_class, random_state=42)
    balanced_data.append(class_data)

df = pd.concat(balanced_data).reset_index(drop=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle

X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values - 1  # Label 0-25

X_images = X.reshape(-1, 28, 28)

print(f" Dataset : {X_images.shape} | Kelas: {len(np.unique(y))}")

In [ ]:
# Data Visualization
plt.figure(figsize=(12, 6))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_images[i], cmap='gray')
    plt.title(f'Label: {chr(y[i] + ord("A"))}')
    plt.axis('off')
plt.suptitle(' Karakter EMNIST Letters')
plt.show()

In [ ]:
# HOG Feature Extraction
def extract_hog_features(images, orientations=9, pixels_per_cell=(7,7), cells_per_block=(2,2)):
    features = []
    hog_images = []
    for img in images:
        fd, hog_img = hog(img,
                         orientations=orientations,
                         pixels_per_cell=pixels_per_cell,
                         cells_per_block=cells_per_block,
                         block_norm='L2-Hys',
                         transform_sqrt=True,
                         visualize=True)
        features.append(fd)
        hog_images.append(hog_img)
    return np.array(features), hog_images


print("Mengekstrak HOG features...")
X_hog, hog_vis = extract_hog_features(X_images)

print(f"HOG Feature shape: {X_hog.shape}")

# Visualisasi HOG
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(X_images[0], cmap='gray')
plt.title('Original Image')
plt.subplot(1, 2, 2)
plt.imshow(hog_vis[0], cmap='gray')
plt.title('HOG Features')
plt.show()

In [ ]:
#split data
X_train, X_test, y_train, y_test = train_test_split(
    X_hog, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} sampel")
print(f"Test : {X_test.shape[0]} sampel")

In [ ]:
# SVM + GridSearch 
param_grid = {
    'C': [1, 10],
    'gamma': ['scale', 0.01],
    'kernel': ['rbf']
}

svm = SVC(random_state=42)
grid_search = GridSearchCV(svm, param_grid, cv=5, scoring='accuracy', n_jobs=-1)

print("Training model dengan GridSearch...")
grid_search.fit(X_train, y_train)

print("Parameters:", grid_search.best_params_)
best_svm = grid_search.best_estimator_


In [ ]:
# Evaluation
y_pred = best_svm.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=[chr(i + ord('A')) for i in range(26)]))

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}")

#confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[chr(i + ord('A')) for i in range(26)],
            yticklabels=[chr(i + ord('A')) for i in range(26)])

plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.title('Confusion Matrix')
plt.show()